In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_path = r"D:\archive\train"
test_path = r"D:\archive\test"

train_dataset = datasets.ImageFolder(train_path, transform=train_transform)
test_dataset = datasets.ImageFolder(test_path, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Classes:", train_dataset.classes)

model = models.resnet18(pretrained=True)


for name, param in model.named_parameters():
    if "layer4" not in name:  # fine-tune last block
        param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 3)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0005)

# Optional: learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

epochs = 10  

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    
    scheduler.step() 
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
print("Test Accuracy:", acc)

Using device: cpu


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'D:\\archive\\train'

In [7]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio

  Using cached torch-2.10.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/113.8 MB ? eta -:--:--
   ---------------------------------------- 0.8/113.8 MB 2.3 MB/s eta 0:00:49
   ---------------------------------------- 1.0/113.8 MB 2.3 MB/s eta 0:00:50
    --------------------------------------- 1.6/113.8 MB 2.3 MB/s eta 0:00:49
    --------------------------------------- 2.1/113.8 MB 2.3 MB/s eta 0:00:49
    --------------------------------------- 2.6/113.8 MB 2.3 MB/s eta 0:00:48
   - -------------------------------------- 3.1/113.8 MB 2.3 MB/s eta 0:00:48
   - -------------------------------------- 3.7/113.8 MB 2.3 MB/s eta 0:00:49
   - -------------------------------------- 3.9/113.8 MB 2.3 MB/s eta 0:00:49
   - -------------------------------------- 4.5/113.8 MB 2.3 MB/s eta 0:00:49
   - -------------------------------------- 5.0/113.8 MB 2.2 MB/s eta 0:00:49
   - --